## CHURN PREDICTION using Binary Classification

In [1]:
%pip install -q kagglehub pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import kagglehub

path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Downloaded to:", path)

Downloaded to: /home/codespace/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1


In [3]:
import pandas as pd

df = pd.read_csv(f"{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.head().T # Transpose the first 5 rows of the DataFrame to view them in a vertical format.

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [5]:
# Clean up the column names
df.columns = df.columns.str.lower().str.replace(' ','_')

categorical_variables = list(df.dtypes[(df.dtypes == 'str')].index)

# Clean up the body of the table
for c in categorical_variables: 
    df[c] = df[c].str.lower().str.replace(' ','_')

df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [6]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [7]:
tc = pd.to_numeric(df['totalcharges'],errors="coerce")
tc

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

In [8]:
df['totalcharges'] = pd.to_numeric(df['totalcharges'],errors="coerce")
df['totalcharges'] = df['totalcharges'].fillna(0)

In [9]:
df[tc.isnull()][['customerid','totalcharges']]

,customerid,totalcharges
488,4472-lvygi,0.0
753,3115-czmzd,0.0
936,5709-lvoeq,0.0
1082,4367-nuyao,0.0
1340,1371-dwpaz,0.0
3331,7644-omvmy,0.0
3826,3213-vvolg,0.0
4380,2520-sgtta,0.0
5218,2923-arzlg,0.0
6670,4075-wkniu,0.0


In [10]:
df.churn = (df.churn == 'yes').astype(int)

## Validation Framework
Performing the train/val/test with scikit-learn

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
df_full_train, df_test = train_test_split(df,test_size=0.2,random_state=1)
len(df_test), len(df_full_train)

df_train, df_val = train_test_split(df_full_train,test_size=0.25,random_state=1)
len(df_test), len(df_train), len(df_val)

(1409, 4225, 1409)

In [13]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [14]:
y_train = df_train.churn.values 
y_val = df_val.churn.values 
y_test = df_test.churn.values 

df_train = df_train.drop(columns=['churn'])
df_val = df_val.drop(columns=['churn'])
df_test = df_test.drop(columns=['churn'])

In [15]:
df_train.T

,0,1,2,3,4,5,6,7,8,9,...,4215,4216,4217,4218,4219,4220,4221,4222,4223,4224
customerid,8015-ihcgw,1960-uycnn,9250-wypll,6786-obwqr,1328-euzhc,8676-ooqej,1452-voqch,6653-cbbom,5893-kclgt,3992-ywpko,...,3190-fzatl,7734-dboai,0369-zgovk,6937-gcdgq,3589-ppvkw,1309-xgfsn,4819-hjpiw,3703-vavcl,3812-lrzir,1704-nrwye
gender,female,male,female,female,female,male,male,female,female,female,...,male,female,female,male,male,male,male,male,female,female
seniorcitizen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
partner,yes,no,no,yes,yes,no,no,no,no,no,...,no,yes,yes,yes,no,yes,no,yes,yes,no
dependents,yes,no,no,yes,no,no,no,no,yes,no,...,yes,yes,yes,yes,no,yes,no,yes,yes,no
tenure,72,10,5,5,18,4,1,1,72,6,...,14,1,28,19,9,52,18,2,27,9
phoneservice,yes,yes,yes,yes,yes,no,yes,yes,yes,yes,...,yes,no,yes,yes,yes,yes,no,yes,yes,yes
multiplelines,yes,yes,yes,no,no,no_phone_service,yes,no,no,yes,...,no,no_phone_service,no,no,no,yes,no_phone_service,no,yes,no
internetservice,fiber_optic,fiber_optic,fiber_optic,fiber_optic,no,dsl,fiber_optic,fiber_optic,no,fiber_optic,...,no,dsl,fiber_optic,dsl,dsl,dsl,dsl,fiber_optic,no,fiber_optic
onlinesecurity,yes,no,no,no,no_internet_service,no,no,no,no_internet_service,no,...,no_internet_service,no,no,yes,no,no,no,no,no_internet_service,no


## Exploratory Data Analysis
1. check missing values
2. look at the target variable (churn)
3. look at the numerical and categorical variables

In [16]:
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,5442-pptjy,male,0,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
1,6261-rcvns,female,0,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
2,2176-osjuv,male,0,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
3,6161-erdgd,male,0,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
4,2364-ufrom,male,0,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


In [17]:
df_full_train.isnull().sum() ## no missing values found

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [18]:
df_full_train.churn.value_counts() #shows the value counts in the column
## Churn Rate
df_full_train.churn.value_counts(normalize=True) #shows the value counts as a percentage

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [19]:
## for a column with binary variable, you can also just calculate the mean 
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate, 2)

np.float64(0.27)

In [20]:
list(df_full_train.columns)

['customerid',
 'gender',
 'seniorcitizen',
 'partner',
 'dependents',
 'tenure',
 'phoneservice',
 'multiplelines',
 'internetservice',
 'onlinesecurity',
 'onlinebackup',
 'deviceprotection',
 'techsupport',
 'streamingtv',
 'streamingmovies',
 'contract',
 'paperlessbilling',
 'paymentmethod',
 'monthlycharges',
 'totalcharges',
 'churn']

In [21]:
numerical = [
    'tenure', 
    'monthlycharges', 
    'totalcharges'
    ]
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod'
]
#df_full_train.dtypes
df_full_train[categorical].nunique()


gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

## Feature Importance: churn rate & risk ratio
feature important analysis, determines which features affect our target variable

In [26]:
df_full_train.head()
churn_female = df_full_train[df_full_train['gender'] == 'female'].churn.mean()
churn_female

np.float64(0.27682403433476394)

In [28]:
churn_male = df_full_train[df_full_train['gender'] == 'male'].churn.mean()
churn_male, global_churn_rate

(np.float64(0.2632135306553911), np.float64(0.26996805111821087))

In [33]:
df_full_train.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

In [41]:
churn_partner = df_full_train[df_full_train['partner'] == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train['partner'] == 'no'].churn.mean()
churn_partner, churn_no_partner
global_churn_rate-churn_partner, global_churn_rate-churn_no_partner

(np.float64(0.06493474245795922), np.float64(-0.05984095297455855))

In [46]:
churn_no_partner / global_churn_rate, churn_partner / global_churn_rate

(np.float64(1.2216593879412643), np.float64(0.7594724924338315))